# The mathematics of linear regression from scratch

This notebook is the full, in-depth version of the mathematics behind
`linear_regression_from_scratch`. It expands on Part 4 of the README.

Every formula is written in plain ASCII:

- `^` means 'to the power of'
- `*` means 'multiplied by'
- `SUM` means a sum over all the samples
- Greek letters are spelled out (alpha, sigma, etc.)
- `X^T` means the transpose of `X`

It is meant to be read top to bottom. Nothing here requires the library itself:
the goal is to understand exactly what the code in
`src/linear_regression_from_scratch/` is computing, and why.

# What I am building

## The goal

Given training data of inputs `X` and targets `y`, learn parameters `w`
(weights) and `b` (bias) so that the prediction `y_hat` is as close as
possible to the true value `y`.

For a single input `x` this is the equation of a line:

```
y_hat = w * x + b
```

For many inputs at once we use a matrix `X` (one row per sample, one column
per feature):

```
y_hat = X * w + b
```

Throughout, `m` is the number of samples (rows of X) and `n` is the number of
features (columns of X).

## Three components

Every model in the library reduces to three pieces:

1. **Prediction** - turn inputs into a guess (`y_hat`).
2. **Cost function** - measure how wrong the guess is.
3. **Optimization** - change `w` and `b` to make the cost smaller.

The three components below are the heart of the whole library.

### Component 1 - Prediction (Forward Pass)

```
y_hat = X * w + b
```

Here `X * w` is a matrix-vector product. For sample `i` that means:

```
y_hat_i = x_i . w + b
```

Input flows through the model to produce predictions. This is the only line
that produces an output; everything else exists to make this output correct.
Implemented in `base.py` (the `predict` method).

### Component 2 - Cost Function (Error Measurement)

We measure error with mean squared error:

```
J(w, b) = (1 / (2 * m)) * SUM_i (y_hat_i - y_i)^2
```

- Squaring makes every error positive (so positive and negative errors do not
  cancel out) and it penalizes large errors more than small ones.
- Dividing by `m` makes the cost independent of the dataset size.
- The extra `1/2` is a convenience: it cancels the `2` that appears when we
  differentiate `(y_hat - y)^2`, which keeps the gradient formulas clean.

The term `(y_hat_i - y_i)` is the error for sample `i`. Note the order: it is
`y_hat - y`, not `y - y_hat`. Getting this sign backwards is the most common
silent bug in a from-scratch implementation. Implemented in `base.py`
(the `_cost` method).

### Component 3 - Optimization (Learning)

We change the parameters in the direction that reduces the cost. The gradient
points the way the cost increases fastest, so we step in the opposite
direction:

```
w <- w - alpha * dJ/dw
b <- b - alpha * dJ/db
```

Iteratively adjusts the parameters to reduce the cost. `alpha` is the
**learning rate** (the step size). Too small and learning is painfully slow;
too large and the cost can bounce around or even grow instead of falling.

## Gradient with respect to w

We want `dJ/dw`. The cost depends on `w` only through `y_hat`, and
`y_hat_i = x_i . w + b`, so the derivative of `y_hat_i` with respect to a
single weight `w_j` is just `x_ij` (the j-th feature of sample i). Applying
the chain rule component by component:

```
dJ/dw_j = (1/(2m)) * SUM_i  2 * (y_hat_i - y_i) * x_ij
        = (1/m)  * SUM_i  (y_hat_i - y_i) * x_ij
```

The leading `2` cancels the `1/2` in the cost - that is the only reason the
`1/2` is there. Stacking all the `w_j` into one vector and recognising the
sum as a matrix-vector product gives the compact vector form:

```
dJ/dw = (1/m) * X^T * (y_hat - y)
      = (1/m) * X^T * (X*w + b - y)
```

`X^T` is the transpose of `X`. Implemented in `base.py` (the `_gradient`
method).

## Gradient with respect to b

The bias `b` appears in every `y_hat_i` with coefficient 1, so the derivative
of `y_hat_i` with respect to `b` is 1. The same chain rule gives:

```
dJ/db = (1/(2m)) * SUM_i  2 * (y_hat_i - y_i) * 1
      = (1/m)  * SUM_i  (y_hat_i - y_i)
      = (1/m)  * sum(y_hat - y)
```

In words: the gradient of the bias is just the average error over all
samples.

### Numerical sanity check

Before trusting an analytical gradient, it is good practice to check it
against the definition of the derivative - the finite-difference estimate.
The cell below computes the gradients derived above and compares them with a
central-difference estimate of the cost. If they agree, the derivations are
correct.

In [ ]:
# Numerical check: do the analytical gradients match a finite-difference estimate?
import numpy as np

rng = np.random.default_rng(0)
X = rng.uniform(0, 10, size=(50, 2))
true_w = np.array([3.0, -2.0]); true_b = 4.0
y = X @ true_w + true_b + rng.normal(0, 1, 50)

# a test point (not the optimum)
w = np.array([1.0, 1.0]); b = 0.0
m = len(y)
y_hat = X @ w + b

# analytical gradient (derived above)
dw = (X.T @ (y_hat - y)) / m
db = np.sum(y_hat - y) / m

# numerical gradient of the cost J = (1/(2m)) * sum (y_hat - y)^2
def cost(w, b):
    return np.mean((X @ w + b - y) ** 2) / 2

eps = 1e-6
dw_num = np.array([
    (cost(w + eps * e, b) - cost(w - eps * e, b)) / (2 * eps)
    for e in np.eye(2)
])
db_num = (cost(w, b + eps) - cost(w, b - eps)) / (2 * eps)

print('analytical dw:', dw)
print('numerical  dw:', dw_num)
print('analytical db:', db)
print('numerical  db:', db_num)
print('match:', np.allclose(dw, dw_num) and np.allclose(db, db_num))

`match: True` confirms that the hand-derived gradients match the brute-force
derivative. This is the same idea behind the gradient-check test in
`tests/test_gradients.py`.

## Ridge regression (L2 regularization)

Ridge adds a penalty proportional to the squared size of the weights:

```
J_ridge = (1/(2m)) * SUM (y_hat - y)^2  +  (alpha/2) * SUM w_j^2
```

This penalizes large weights and shrinks them toward zero. Shrinking reduces
overfitting: the model cannot lean too heavily on any single feature, so it
tends to generalize better to new data.

Only the weight gradient changes - it gains a `+ alpha * w` term:

```
dJ/dw = (1/m) * X^T * (y_hat - y) + alpha * w
dJ/db = (1/m) * SUM (y_hat - y)        # the bias is NOT penalized
```

The bias is deliberately left out of the penalty: shrinking an intercept has
no useful meaning. Larger `alpha` means more shrinkage. Implemented in
`models.py` (`RidgeRegression._penalty_gradient`).

## Lasso regression (L1 regularization) and feature selection

Lasso adds a penalty proportional to the absolute size of the weights:

```
J_lasso = (1/(2m)) * SUM (y_hat - y)^2  +  alpha * SUM |w_j|
```

The absolute value `|w_j|` is **not differentiable at zero**, so we cannot
just write down `dJ/dw` and add it to the gradient. Instead we use
**proximal gradient descent** (also called ISTA), in two steps:

1. Take a normal gradient step on the smooth (squared-error) part:

```
w <- w - step * (1/m) * X^T * (y_hat - y)
```

2. Then apply the **soft-thresholding** operator, which is the proximal map
   of the L1 penalty:

```
soft_threshold(x, lambda) = sign(x) * max(|x| - lambda, 0)
```

   where the threshold is `lambda = step * alpha` (step size times penalty
   strength).

Soft-thresholding pulls every weight toward zero by `lambda`, and - crucially
- sets it to **exactly zero** if it crosses. That exact zeroing is Lasso's
famous **automatic feature selection**: features whose true influence is zero
or near zero get dropped from the model entirely. Implemented in `models.py`
(`LassoRegression._prox`) and `solvers.py` (`soft_threshold`).

## The closed-form solver (normal equations)

Linear regression also has an exact, one-step solution that needs no gradient
descent:

```
w = (X^T * X)^(-1) * X^T * y
```

This is the **normal equation**. It is exact and fast, which is why the
library offers it as `solver='normal'`.

The catch: explicitly computing `(X^T * X)^(-1)` is numerically unstable when
features are correlated (collinear). Then `X^T * X` is nearly singular, its
condition number explodes, and the inverse amplifies rounding errors into
garbage weights. The library therefore avoids the explicit inverse and uses
the **pseudo-inverse** through least squares (`np.linalg.lstsq`), which is
based on the SVD (singular value decomposition). The SVD handles
rank-deficient and collinear matrices gracefully instead of dividing by
near-zero. Implemented in `base.py` (`_fit_normal`).

## Feature standardization

Gradient descent is very sensitive to the scale of the inputs. If one feature
ranges in the thousands and another in the tenths, the cost surface becomes a
long, thin valley and gradient descent crawls along it slowly and unstably.

To fix this, inputs are standardized internally to zero mean and unit standard
deviation:

```
z = (x - mean) / std
```

The model then learns weights in this standardized space:
`y_hat = w . z + b`. But the user wants coefficients in original units, so we
convert back. Expanding the standardized prediction:

```
y = w . ((x - mean) / std) + b
  = (w / std) . x + (b - (w / std) . mean)
```

So the original-space parameters are:

```
w_raw = w / std
b_raw = b - SUM (w_raw * mean)
```

A subtle but important point: the bias absorbs a correction term
`SUM (w_raw * mean)`. Forgetting that term leaves the slope correct but the
intercept wrong. Implemented in `preprocessing.py`
(`StandardScaler.unstandardize_weights`).

## The optimizers (solvers)

All update rules share one engine (`solvers.py`, the `minimize` function).
They differ only in how they turn the gradient into a step.

**batch** - vanilla gradient descent, the full dataset every step. Stable but
slow, because each step costs a pass over all `m` samples:

```
w <- w - alpha * dJ/dw
```

**sgd** (stochastic gradient descent) - one randomly chosen sample per step.
Cheap and quick to make early progress, but noisy: each step is only a rough
estimate of the true gradient.

**minibatch** - a small random subset (say 32 samples) per step. The practical
default that balances the stability of batch with the speed of sgd.

**momentum** - keeps a running velocity that accumulates the gradient, so
consistent directions speed up and noisy ones cancel out. This pushes through
flat regions (plateaus):

```
v <- momentum * v - alpha * dJ/dw
w <- w + v
```

**adam** - adaptive moment estimation. It keeps running estimates of the
gradient's first moment `m` (its mean) and second moment `s` (its uncentered
variance), and gives each parameter its own step size. The `1 - beta^t` terms
are bias corrections that account for the estimates starting at zero (without
them the early steps explode):

```
m <- beta1 * m + (1 - beta1) * dJ/dw
s <- beta2 * s + (1 - beta2) * (dJ/dw)^2
w <- w - alpha * (m / (1 - beta1^t)) / (sqrt(s / (1 - beta2^t)) + eps)
```

Typical defaults are `beta1 = 0.9`, `beta2 = 0.999`, `eps = 1e-8`. Adam is
usually the fastest to converge and the least sensitive to the learning rate,
which is why the README quick-start uses it.

## Early stopping

There is no point iterating once the cost has effectively stopped falling. The
check compares the cost to the previous iteration. Rather than an absolute
threshold (which breaks when costs are large or tiny), it uses a **relative**
test against a running scale:

```
if |cost_prev - cost_now| <= tol * max(1.0, |cost_prev|):
    stop
```

The `max(1.0, |cost_prev|)` keeps the test well-behaved as the cost shrinks
toward zero. `tol` is the tolerance. Implemented in `solvers.py` (the
`minimize` loop) and controlled by the `tol` parameter in `base.py`.

## Evaluation metrics

Once the model is fitted, we score it. Three metrics are provided.

**R2 (coefficient of determination)** - the fraction of the variance in `y`
explained by the model. 1 is a perfect fit, 0 is 'no better than guessing the
mean', and it can even go negative for a model worse than that:

```
R2 = 1 - SUM(y - y_hat)^2 / SUM(y - mean(y))^2
```

**RMSE (root mean squared error)** - the typical size of an error, in the same
units as the target. Lower is better. Because it squares first, it is
sensitive to large errors:

```
RMSE = sqrt(mean((y - y_hat)^2))
```

**MAE (mean absolute error)** - the average absolute error. Lower is better.
Less sensitive to outliers than RMSE, so it gives a steadier view of the
typical error:

```
MAE = mean(|y - y_hat|)
```

Implemented in `base.py` (`score`, `rmse`, `mae`).

## Summary of the whole loop

Putting the pieces together, training one model is:

```
1. Standardize X (internally).
2. Predict:     y_hat = X * w + b
3. Cost:        J = (1/(2m)) * SUM (y_hat - y)^2   (+ penalty)
4. Gradient:    dJ/dw, dJ/db   (with regularization added)
5. Update:      w <- w - alpha * dJ/dw ,  b <- b - alpha * dJ/db
                (+ soft-threshold step for Lasso)
6. Repeat 2-5 until the early-stopping test fires.
7. Un-standardize w, b back to original units.
```

Ridge and Lasso plug in only at steps 4 and 5; everything else is shared.
That is exactly why the library can implement three different models with one
engine. It is also why the same `solvers.py` will carry over directly to the
next project, neural-networks-from-scratch.